# Figure2(Stackedbar) for bertopic

In [1]:
visualization_target = input("INPUT 'visualization_target RUN_ID'(e.g., run_id_14): ")
model_in_run = input("INPUT 'model_in_run'(e.g., bert_based, lda, tag): ")

In [2]:
import os 
import numpy as np
import pandas as pd
from datetime import datetime
import pprint

from setting_for_sda.color_setting import Color_Setting
from setting_for_sda.date_setting import Date_Setting
from setting_for_sda.path_setting import path_list

import lib.stats.stats as st
from lib.utils.statistics import *
from matplotlib import pyplot as plt

import matplotlib as mpl
import lib.visualization.plot_generator as PlotGen
import lib.visualization.font_setting as font_setting
from lib.visualization.distribution_collector import (proportion_calc_for_topic, 
                                                      collect_top_bottom_topic)
mpl.rcParams['font.family'] = font_setting.init_font()

Helvetica /home/mghan/.fonts/Helvetica/Helvetica Oblique.ttf
Registered font name: Helvetica


In [3]:
viz_dir = f'{path_list["data_root_dir"]}/result/{model_in_run}/{visualization_target}'
data_dir = f"{viz_dir}/data"
option_dict = load_json(f"{viz_dir}/option.json")

output_dir = './fig/'
date_range = 'Weekly'

std_date = Date_Setting[option_dict['year_range']]['std_date']

pprint.pprint(option_dict)

{'data_dir': '/mnt/hdd/mghan/so_data_availability/data/public_for_260105/questions/python/2021to2025',
 'model': 'BERTopic',
 'model_option': {'clustering': {'n_clusters': 50, 'name': 'kmeans'},
                  'embedding_model': None,
                  'model_type': 'text',
                  'nr_topics': None,
                  'vectorizer': 'CountVectorizer'},
 'run_id': '14',
 'save_dir': '/mnt/hdd/mghan/so_data_availability/result/bert_based/run_id_14',
 'selected_tags': None,
 'snapshot': 'public_for_260105',
 'visualization': {'n': 10},
 'year_range': '2021to2025'}


In [4]:
df = load_df(data_dir, ['id' , 'creationdate' , 'title', 'tags', 'body', 'Topic'])

In [5]:
df['creationdate'] = pd.to_datetime(df['creationdate'], format="mixed").dt.normalize()
df['rel_week'] = ((df['creationdate'] - std_date).dt.days // 7)

In [6]:
print(df['creationdate'].min())
print(df['creationdate'].max())

2021-12-01 00:00:00
2025-11-25 00:00:00


In [7]:
prop_df = proportion_calc_for_topic(df, 'rel_week', 'Topic')

In [8]:
top10list, mid30list, bot10list = collect_top_bottom_topic(prop_df, 'rel_week', 'Topic', 'proportion')

list_10 = {'Top 20% Topics' : top10list, 
               'Bottom 20% Topics' : bot10list}


df_dict = {'Top 20% Topics'       : prop_df[prop_df['Topic'].isin(top10list)],   
          'Bottom 20% Topics'     : prop_df[prop_df['Topic'].isin(bot10list)]}

    


In [9]:
df[df['Topic'].isin(top10list)]

,id,creationdate,title,tags,body,Document,Topic,rel_week
2,75460949,2023-02-15,Modify an html text using bs4,<python><html><web-scraping><beautifulsoup>,<p>I'm writing an script to translate the visi...,I'm writing an script to translate the visible...,8,11
6,75495676,2023-02-18,How to write csv file in descending order in p...,<python><csv><dictionary>,<pre><code>ProductMaster = { 1 : [Minor Widget...,\n\n\n\nI got the dictionary (ProductMaster) a...,6,11
7,75460996,2023-02-15,"After brew install of python@3.9 ""python"" is n...",<python>,<p><code>python3</code> is available after:</p...,python3 is available after:\n\n\n\n\nBut there...,2,11
8,75408737,2023-02-10,Tried to install python 3.7.0 got build failed,<python><ubuntu><pyenv>,<p>Iam trying to install python 3.7.0 in ubunt...,Iam trying to install python 3.7.0 in ubuntu u...,2,10
9,75497443,2023-02-19,Exception in callback AsyncIOScheduler.wakeup,<python><pyqt><python-asyncio>,<p>Working with a PyQt application that uses t...,Working with a PyQt application that uses the ...,0,11
...,...,...,...,...,...,...,...,...
422023,71388681,2022-03-08,How to make weighted list with list comprehension,<python><python-3.x><list><list-comprehension>,<p>I have a list <code>w</code> where <code>w[...,I have a list w where w[i] represents the numb...,6,-39
422027,71285681,2022-02-27,Is there a way to put JSON data to django models?,<python><django>,<p>So what I want to do here is that take a CS...,So what I want to do here is that take a CSV f...,3,-40
422028,71285795,2022-02-27,How do I fix this I want to solve it using a w...,<python><while-loop>,<pre><code>num = 0\nnumber = int (input(&quot;...,\n\n\n\n**I want it to print word as many time...,4,-40
422029,71285832,2022-02-27,How to insert Python Variables into HTML?,<python><html><pyqt5>,"<p><a href=""https://i.sstatic.net/JLGJ7.png"" r...","My intention is to set a QLabel Text, bypassin...",0,-40


In [10]:
rel_week_list = sorted(df.loc[df['Topic'].isin(bot10list), 'rel_week'].unique())

In [11]:
import lib.database.DBInterface as db_interface
db_if = db_interface.DBInterface()

for rel_week in rel_week_list :
    tmp = df.loc[df['Topic'].isin(top10list)].copy()
    tmp = tmp[tmp['rel_week'] ==rel_week]
    tmp = tmp[tmp['creationdate'].dt.day.isin([15, 25])]
    dt_p_id_list = tmp[['creationdate', 'id']].values

    c_sql = """select nextval(%s);""" 
    rows = db_if.execute_query(c_sql, ('seq_sample_ver_3333',))
    var = rows[0]

    first_ann_q_id = [[int(var[0]),date , int(id)] for date, id in dt_p_id_list]
    sql = f'INSERT INTO tt_posts_difficulty_target  VALUES %s'
    db_if.execute_bulk_values(sql, first_ann_q_id)  
    



Bulk insert executed successfully.
Bulk insert executed successfully.
Bulk insert executed successfully.
Bulk insert executed successfully.
Bulk insert executed successfully.
Bulk insert executed successfully.
Bulk insert executed successfully.
Bulk insert executed successfully.
Bulk insert executed successfully.
Bulk insert executed successfully.
Bulk insert executed successfully.
Bulk insert executed successfully.
Bulk insert executed successfully.
Bulk insert executed successfully.
Bulk insert executed successfully.
Bulk insert executed successfully.
Bulk insert executed successfully.
Bulk insert executed successfully.
Bulk insert executed successfully.
Bulk insert executed successfully.
Bulk insert executed successfully.
Bulk insert executed successfully.
Bulk insert executed successfully.
Bulk insert executed successfully.
Bulk insert executed successfully.
Bulk insert executed successfully.
Bulk insert executed successfully.
Bulk insert executed successfully.
Bulk insert executed